### File containing biosamples already mined for UHVDB

In [ ]:
### cnGVC
# https://static-content.springer.com/esm/art%3A10.1186%2Fs13073-025-01460-6/MediaObjects/13073_2025_1460_MOESM2_ESM.xlsx

### mMGE
# https://oup.silverchair-cdn.com/oup/backfile/Content_public/Journal/nar/49/D1/10.1093_nar_gkaa869/1/gkaa869_supplemental_file.xls?Expires=1781823570&Signature=QOkvcOL2AFyBuzY3l5MaracfARm1~03ns2sv8ADxoyz43~q0UqO2AV~wCfIBhcbC2qwnzReTbb-sWlNZQnR1Xl1LINlv7hPcnTo7seBhv7m4GDYKGt5t4msnPobYLKTtFQgf4mcYFsnXR5lJXqv0-ZTUTrQxszk~UQcxCTCF9yPOaB21hoUgweyXvN9H3h25t-8zmpksJW7pgXZcVB2lHZSz6CUMOhqv24o~V0YfmDbd2hBSH-5AJVQUR9CmYApu0jzQu~0kI3Niwj0H3TD6NScZGQjOS7pmsyhj2PNBAD76XTwZvVziyqN2PFhOyvinQ3E1cFE4dS13Kgmzdd15HA__&Key-Pair-Id=APKAIE5G5CRDK6RD3PGA

### UHGV

### IMGVR

### CNGVR

### CHVD

### 

### Example script for creating ENA samplesheets

In [ ]:
### ENA human airways, skin, and urogenital samples

# download ena metagenome metadata
!curl -X 'POST' \
'https://www.ebi.ac.uk/ena/portal/api/search' -H 'accept: */*' -H 'Content-Type: application/x-www-form-urlencoded' \
-d 'excludeAccessionType=&download=false&query=&excludeAccessions=&includeMetagenomes=true&dataPortal=metagenome&searchCurations=false&includeAccessionType=&includeAccessions=&format=tsv&fields=analysis_accession%2Csample_accession%2Crun_accession%2Cscientific_name%2Cbroad_scale_environmental_context%2Cenvironment_biome%2Chost_body_site%2Cgenerated_ftp%2Cfirst_public&dccDataOnly=false&rule=&result=analysis' \
> ena_metagenomes.tsv

# load ena metagenome metadata
import polars as pl
ena_meta = pl.read_csv('ena_metagenomes.tsv', separator='\t')

# create a samplesheet of airways, skin, and urogenital samples
ena_human_samples = (
    ena_meta
        .filter(
            (pl.col('scientific_name').str.contains('oral')) |
            (pl.col('scientific_name').str.contains('nasopharyngeal')) |
            (pl.col('scientific_name').str.contains('lung')) |
            (pl.col('scientific_name').str.contains('saliva ')) |
            (pl.col('scientific_name').str.contains('skin')) |
            (pl.col('scientific_name').str.contains('vaginal ')) |
            (pl.col('scientific_name').str.contains('urinary'))
        )
)

(
    ena_human_samples
        .rename({
            'analysis_accession':'sample',
            'sample_accession':'biosample',
            'run_accession':'acc'
        })
        .with_columns([
            pl.when(pl.col('scientific_name').str.contains(r'oral|nasopharyngeal|lung|saliva'))
                .then(pl.lit('Airways'))
                .when(pl.col('scientific_name').str.contains('skin'))
                .then(pl.lit('Skin'))
                .when(pl.col('scientific_name').str.contains(r'vaginal|urinary|urogenital'))
                .then(pl.lit('Urogenital'))
                .when(pl.col('scientific_name').str.contains(r'gut|feces|stool|fecal'))
                .then(pl.lit('Gut'))
                .otherwise(pl.lit('Unknown'))
                .alias('body_site'),
            pl.lit('Assembly').alias('db_type'),
            (pl.lit('https://') + pl.col('generated_ftp')).alias('fasta'),
        ])
        .drop('generated_ftp')
        .write_csv('ena_samplesheet.csv')
)

# count number of ENA samples
print(f"Number of human-associated ENA metagenome assemblies: {ena_human_samples.height}")

### Example script for creating Logan samplesheet

In [ ]:
### Logan human airways, skin, and urogenital samples

# parse SRA metadata to find human-associated metagenomes
import duckdb

duckdb.sql('''
INSTALL httpfs;
LOAD httpfs;
INSTALL parquet;
LOAD parquet;
COPY (
  SELECT 
    acc, assay_type, consent, libraryselection, librarysource, organism
  FROM read_parquet('s3://sra-pub-metadata-us-east-1/sra/metadata/*')
  WHERE assay_type != 'AMPLICON'
  AND consent = 'public'
  AND libraryselection != 'PCR'
  AND (librarysource = 'METAGENOMIC'
    OR organism LIKE '%microbiom%'
    OR organism LIKE '%metagenom%')
) TO '2025_08_29_sra_metadata.parquet' (FORMAT 'parquet');
''')

import polars as pl

# identify top organisms
sra_meta = (
    pl.read_parquet('sra_metadata.parquet',
        columns=['acc', 'organism']
    )
)

# identify human-associated metagenomes
human_organisms = set([
    'human oral metagenome',
    'human nasopharyngeal metagenome',
    'oral metagenome',
    'human lung metagenome',
    'respiratory tract metagenome',
    'human saliva metagenome',
    'upper respiratory tract metagenome',
    'human sputum metagenome',
    'lung metagenome',
    'human tracheal metagenome',
    'oral-nasopharyngeal metagenome',
    'human skin metagenome',
    'skin metagenome',
    'human vaginal metagenome',
    'vaginal metagenome',
    'human urinary tract metagenome',
    'urinary tract metagenome',
    'human reproductive system metagenome',
    'reproductive system metagenome'
])

sra_meta_human = (
    sra_meta
        .filter(
            (pl.col('organism').is_in(human_organisms))
        )
)

# identify samples in logan
!wget https://s3.amazonaws.com/logan-pub/stats/logan-seqstats-contigs-v1.1.parquet
logan_stats = pl.read_parquet('logan-seqstats-contigs-v1.1.parquet')
sra_meta_human_logan = (
    sra_meta_human
        .filter(pl.col('acc').is_in(logan_stats['accession']))
)
print(f"Number of human-associated metagenomes in Logan: {sra_meta_human_logan.height}")

# create samplesheet for mining logan metagenomes
(
    sra_meta_human_logan
        .with_columns([
            pl.lit('LOGAN').alias('source_db'),
            (pl.lit('s3://logan-pub/c/') + pl.col('acc') + '/' + pl.col('acc') + '.contigs.fa.zst').alias('fasta'),
            pl.lit(None).alias('tar')
        ])
        .write_csv('logan_samplesheet.csv')
)

### Example script for creating SPIRE samplesheet

In [ ]:
### SPIRE human airways, skin, and urogenital samples

import polars as pl

# load spire microontoloy
spire_ont = (
    pl.read_csv("https://swifter.embl.de/~fullam/spire/metadata/spire_v1_microntology.tsv.gz", separator="\t", has_header=False)
    .with_columns(
        pl.col("column_1").str.splitn(by=" ", n=3)
        .struct.rename_fields(["biosample", "bioproject", "ontology"])
        .alias("fields")
    ).unnest("fields")
)

# filter to human airways, skin, and urogenital samples
spire_human = (
    spire_ont
        .filter(
            (pl.col("ontology").str.contains("animal host")) &
            (
                (pl.col("ontology").str.contains("mouth")) |
                (pl.col("ontology").str.contains("airway")) |
                (pl.col("ontology").str.contains("urogenital")) |
                (pl.col("ontology").str.contains("skin"))
            )
        )
        .unique('biosample')
)

# create samplesheet
(
    spire_human
        .with_columns([
            pl.lit(None).alias('acc'),
            pl.lit('SPIRE').alias('source_db'),
            (pl.lit('https://spire.embl.de/download_assembly/') + pl.col('biosample')).alias('fasta'),
            pl.lit(None).alias('tar'),
            pl.col('biosample').alias('sample'),
        ])
        .drop('bioproject', 'ontology', 'column_1')
).write_csv('spire_samplesheet.csv')

# count number of SPIRE samples
print(f"Number of human-associated metagenomes in SPIRE: {spire_human.height}")

### Example script for creating NCBI Virus samplesheet

In [ ]:
### Create a samplesheet for NCBI Virus
import polars as pl

# download NCBI Virus assembly summary
!wget https://ftp.ncbi.nlm.nih.gov/genomes/genbank/viral/assembly_summary.txt
# fix header line
!sed -i -e 's/#assembly_accession/assembly_accession/g' assembly_summary.txt

ncbi_virus = (
    pl.read_csv('assembly_summary.txt', separator='\t', comment='#', columns=['assembly_accession', 'biosample', 'bioproject', 'genome_rep', 'ftp_path', 'assembly_level', "excluded_from_refseq", "genome_size", "replicon_count", "scaffold_count"])
)
ncbi_virus = (
    ncbi_virus
        .filter(
            (pl.col('assembly_level').is_in(['Complete Genome', 'Chromosome'])) &
            (pl.col('scaffold_count') == 1) &
            (pl.col('genome_rep') == 'Full') &
            (~pl.col('bioproject').is_in(["PRJNA393166", "PRJNA492716", "PRJEB23154", "PRJNA579886", "PRJDB11069"])) &
            ((pl.col('excluded_from_refseq') == 'na') |
            (pl.col('excluded_from_refseq') == 'derived from metagenome; genus undefined')) &
            (pl.col('genome_size') >= 1500)
        )
        .rename({'assembly_accession':'sample', 'ftp_path':'fasta'})
        .with_columns([
            pl.lit('NCBI_VIRUS').alias('source_db'),
            pl.col('biosample').alias('sample'),
            pl.col('fasta') + '/' + pl.col('fasta').str.rpartition('/')[2].str.rpartition('_')[0] + '_genomic.fna.gz'

        ])
        [['sample', 'biosample', 'acc', 'fasta', 'source_db']]
        .write_csv('ncbi_virus_samplesheet.csv')
)

ncbi_virus_r2024_11_samplesheet['fasta'] = ncbi_virus_r2024_11_samplesheet['fasta'] + '/' + ncbi_virus_r2024_11_samplesheet['fasta'].str.rpartition('/')[2] + '_genomic.fna.gz'
ncbi_virus_r2024_11_samplesheet[['acc', 'tar']] = ''
ncbi_virus_r2024_11_samplesheet['source_db'] = 'NCBI_VIRUS'
ncbi_virus_r2024_11_samplesheet['release'] = 'r2024_11'
ncbi_virus_r2024_11_samplesheet[['sample', 'biosample', 'acc', 'fasta', 'tar', 'source_db', 'release']].to_csv('ncbi_virus/r2024_11/samplesheet.csv', index=False)

!gzip ncbi_virus/r2024_11/samplesheet.csv
!rm assembly_summary.txt